In [6]:
import random
import math

class KMeansScratch:
    def __init__(self, k=3, max_iters=100):
        self.k = k
        self.max_iters = max_iters
        self.centroids = []
        self.clusters = [] # Will store list of indices for each cluster

    def fit(self, data):
        """
        Input: data is a list of lists. [[x1, y1], [x2, y2], ...]
        """
        # Step 1: Initialize Centroids
        # We pick 'k' random points from the dataset as starting centers
        self.centroids = random.sample(data, self.k)

        for i in range(self.max_iters):
            # Step 2: Assignment (The Expectation)
            # Create k empty buckets
            self.clusters = [[] for _ in range(self.k)]
            
            for point_idx, point in enumerate(data):
                # Find closest centroid
                closest_centroid_idx = self._get_closest_centroid(point)
                self.clusters[closest_centroid_idx].append(point)

            # Step 3: Update (The Maximization)
            prev_centroids = self.centroids[:] # Copy current centroids
            
            # Calculate new mean for each cluster
            new_centroids = []
            for cluster in self.clusters:
                # If a cluster is empty (rare but possible), keep old centroid
                if not cluster:
                    # Logic: Find which index this empty cluster corresponds to 
                    # and keep the previous one. For simplicity in hardcore mode:
                    # We just re-append a random point to restart it.
                    new_centroids.append(random.choice(data)) 
                else:
                    new_centroids.append(self._calculate_mean(cluster))
            
            self.centroids = new_centroids

            # Step 4: Convergence Check
            # If centroids didn't move (distance is 0), we stop.
            if self._has_converged(prev_centroids, self.centroids):
                print(f"Converged at iteration {i}")
                break

    def predict(self, data):
        predictions = []
        for point in data:
            predictions.append(self._get_closest_centroid(point))
        return predictions

    # --- THE MATH HELPERS ---

    def _get_closest_centroid(self, point):
        min_dist = float('inf')
        closest_idx = -1
        
        for idx, centroid in enumerate(self.centroids):
            dist = self._euclidean_distance(point, centroid)
            if dist < min_dist:
                min_dist = dist
                closest_idx = idx
        return closest_idx

    def _euclidean_distance(self, p1, p2):
        # Sqrt( sum( (x2 - x1)^2 ) )
        # Since these are lists, we iterate by index
        sum_sq_diff = 0
        dimensions = len(p1)
        for i in range(dimensions):
            sum_sq_diff += (p1[i] - p2[i]) ** 2
        return math.sqrt(sum_sq_diff)

    def _calculate_mean(self, points):
        # Points is a list of lists: [[1, 2], [3, 4], ...]
        # We need average X and average Y
        n_points = len(points)
        n_dims = len(points[0])
        
        # Initialize mean point as [0, 0, ...]
        mean_point = [0] * n_dims
        
        # Sum up all coordinates
        for p in points:
            for i in range(n_dims):
                mean_point[i] += p[i]
        
        # Divide by number of points
        for i in range(n_dims):
            mean_point[i] = mean_point[i] / n_points
            
        return mean_point

    def _has_converged(self, old_centroids, new_centroids):
        # Check if the total distance between old and new positions is negligible
        total_movement = 0
        for i in range(len(old_centroids)):
            total_movement += self._euclidean_distance(old_centroids[i], new_centroids[i])
        
        return total_movement == 0

In [7]:
# --- Usage Example ---
if __name__ == "__main__":
    # Dummy Data (2 groups obvious to the human eye)
    # Group 1: Small numbers (0-10)
    # Group 2: Big numbers (100-110)
    X = [
        [2, 3], [3, 3], [1, 2], [2, 1], # Cluster A
        [100, 102], [101, 101], [105, 107], [103, 103] # Cluster B
    ]

    print("--- Hardcore K-Means Training ---")
    model = KMeansScratch(k=2)
    model.fit(X)

    print("\nFinal Centroids:")
    for i, c in enumerate(model.centroids):
        print(f"Cluster {i}: {c}")

    print("\nPredictions:")
    preds = model.predict(X)
    print(preds) # Should be something like [0, 0, 0, 0, 1, 1, 1, 1]

--- Hardcore K-Means Training ---
Converged at iteration 1

Final Centroids:
Cluster 0: [102.25, 103.25]
Cluster 1: [2.0, 2.25]

Predictions:
[1, 1, 1, 1, 0, 0, 0, 0]
